In [ ]:
import os
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("paper", font_scale=1.2)
sns.set_style("whitegrid")

OUTDIR = "/ictstr01/home/icb/dominik.klein/git_repos/ot_pert_new/fig_2/revision/cell_eval"

In [ ]:
# Load per-CT per-perturbation results
indir = "/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/metrics_new_donor_cell_eval_per_ct"

df_per_pert = pl.read_csv(
    os.path.join(indir, "cell_eval_per_ct_aggregated_per_perturbation.csv"),
    infer_schema_length=10000,
    schema_overrides={"split_or_wandb": pl.Utf8},
).to_pandas()

print(f"Per-perturbation shape: {df_per_pert.shape}")
print(f"Methods: {sorted(df_per_pert['method'].unique())}")
print(f"Donors: {sorted(df_per_pert['donor'].unique())}")
print(f"Cell types: {sorted(df_per_pert['cell_type_new'].unique())}")
print(f"Counts per method:")
print(df_per_pert.groupby('method').size())

In [ ]:
# Map method names to display names
method_display = {
    "cellflow": "CellFlow",
    "mean_model_1": "Mean model 1",
    "mean_model_2": "Mean model 2",
    "identity": "Identity",
    "closest_embedding": "Closest embedding",
}

color_dict = {
    "CellFlow": "#B12F8C",
    "Mean model 1": "#8F97A8",
    "Mean model 2": "#566573",
    "Identity": "#BDBDBD",
    "Closest embedding": "#E0E0E0",
}

method_order = ["CellFlow", "Closest embedding", "Mean model 1", "Mean model 2", "Identity"]

df_per_pert["model"] = df_per_pert["method"].map(method_display)

# Average across seeds/splits per (model, donor, perturbation, cell_type_new)
meta_cols = ["model", "donor", "perturbation", "cell_type_new"]
metric_cols = [
    "pearson_delta", "mse_delta", "mae_delta",
    "discrimination_score_l1", "discrimination_score_l2", "discrimination_score_cosine",
    "pearson_edistance", "clustering_agreement",
    "overlap_at_N", "precision_at_N",
    "de_spearman_sig", "de_direction_match", "de_spearman_lfc_sig",
    "de_sig_genes_recall", "pr_auc", "roc_auc",
]

df = df_per_pert.groupby(meta_cols)[metric_cols].mean().reset_index()
print(f"Averaged shape: {df.shape}")

# Per cell-type boxplots (one plot per cell type)

In [ ]:
# All 16 metrics to plot per cell type
all_metrics = [
    # Distribution metrics
    ("pearson_delta", "Pearson (delta)"),
    ("mse_delta", "MSE (delta)"),
    ("mae_delta", "MAE (delta)"),
    ("pearson_edistance", "Pearson E-distance"),
    ("clustering_agreement", "Clustering agreement"),
    # Discrimination metrics
    ("discrimination_score_l1", "Discrimination (L1)"),
    ("discrimination_score_l2", "Discrimination (L2)"),
    ("discrimination_score_cosine", "Discrimination (cosine)"),
    # DE metrics
    ("overlap_at_N", "Overlap at N"),
    ("precision_at_N", "Precision at N"),
    ("de_spearman_sig", "DE Spearman (sig)"),
    ("de_direction_match", "DE direction match"),
    ("de_spearman_lfc_sig", "DE Spearman LFC (sig)"),
    ("de_sig_genes_recall", "DE sig genes recall"),
    ("pr_auc", "PR AUC"),
    ("roc_auc", "ROC AUC"),
]

cell_types = sorted(df["cell_type_new"].unique())
os.makedirs(os.path.join(OUTDIR, "per_ct_plots"), exist_ok=True)

for ct in cell_types:
    df_ct = df[df["cell_type_new"] == ct]
    if df_ct.empty:
        continue
    
    fig, axes = plt.subplots(4, 4, figsize=(20, 16))
    axes = axes.flatten()
    for i, (col, title) in enumerate(all_metrics):
        ax = axes[i]
        data = df_ct.dropna(subset=[col])
        if data.empty:
            ax.set_title(f"{title}\n(no data)")
            continue
        available = [m for m in method_order if m in data["model"].unique()]
        sns.boxplot(
            data=data, x="model", y=col, hue="model", order=available,
            hue_order=available, palette=color_dict, ax=ax, showfliers=False,
            legend=False,
        )
        sns.stripplot(
            data=data, x="model", y=col, order=available,
            color=".3", size=2, alpha=0.5, ax=ax,
        )
        ax.set_title(title)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=45)
    
    ct_safe = ct.replace(" ", "_").replace("/", "_")
    fig.suptitle(f"Cell type: {ct}", fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()
    fig.savefig(
        os.path.join(OUTDIR, "per_ct_plots", f"cell_eval_{ct_safe}.png"),
        dpi=150, bbox_inches="tight",
    )
    fig.savefig(
        os.path.join(OUTDIR, "per_ct_plots", f"cell_eval_{ct_safe}.pdf"),
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

# Combined all cell-types plot

In [ ]:
# Combined plot: one panel per metric, cell types on x-axis, hue = method
# First average across perturbations to get one value per (model, donor, cell_type)
df_avg = df.groupby(["model", "donor", "cell_type_new"])[metric_cols].mean().reset_index()

# Sort cell types by abundance (descending number of data points)
ct_counts = df_avg.groupby("cell_type_new").size().sort_values(ascending=False)
ct_order = ct_counts.index.tolist()

fig, axes = plt.subplots(len(all_metrics), 1, figsize=(16, 5 * len(all_metrics)))
for ax_idx, (col, title) in enumerate(all_metrics):
    ax = axes[ax_idx]
    data = df_avg.dropna(subset=[col])
    if data.empty:
        ax.set_title(f"{title} (no data)")
        continue
    available_cts = [ct for ct in ct_order if ct in data["cell_type_new"].unique()]
    available_methods = [m for m in method_order if m in data["model"].unique()]
    sns.boxplot(
        data=data, x="cell_type_new", y=col, hue="model",
        order=available_cts, hue_order=available_methods,
        palette=color_dict, ax=ax, showfliers=False,
    )
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=60)
    ax.legend(loc="upper right", fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(OUTDIR, "cell_eval_per_ct_all_metrics.png"), dpi=150, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "cell_eval_per_ct_all_metrics.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Compact combined: bar plot showing mean per cell type, one panel per metric (4x4 grid)
fig, axes = plt.subplots(4, 4, figsize=(28, 20))
axes = axes.flatten()
for i, (col, title) in enumerate(all_metrics):
    ax = axes[i]
    data = df_avg.dropna(subset=[col])
    if data.empty:
        ax.set_title(f"{title} (no data)")
        continue
    available_cts = [ct for ct in ct_order if ct in data["cell_type_new"].unique()]
    available_methods = [m for m in method_order if m in data["model"].unique()]
    sns.barplot(
        data=data, x="cell_type_new", y=col, hue="model",
        order=available_cts, hue_order=available_methods,
        palette=color_dict, ax=ax, errorbar="se",
    )
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=60, labelsize=7)
    if i == 0:
        ax.legend(loc="best", fontsize=7)
    else:
        ax.get_legend().remove()

plt.tight_layout()
fig.savefig(os.path.join(OUTDIR, "cell_eval_per_ct_summary.png"), dpi=150, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "cell_eval_per_ct_summary.pdf"), bbox_inches="tight")
plt.show()

# Heatmap: CellFlow improvement over best baseline per cell type

In [ ]:
# Compute mean metric per (model, cell_type)
df_mean = df_avg.groupby(["model", "cell_type_new"])[metric_cols].mean().reset_index()

# For each metric, compute CellFlow value and best baseline value per cell type
higher_better = {
    "pearson_delta": True, "mse_delta": False, "mae_delta": False,
    "discrimination_score_l1": False, "discrimination_score_l2": False,
    "discrimination_score_cosine": False,
    "pearson_edistance": True, "clustering_agreement": True,
    "overlap_at_N": True, "precision_at_N": True,
    "de_spearman_sig": True, "de_direction_match": True,
    "de_spearman_lfc_sig": True, "de_sig_genes_recall": True,
    "pr_auc": True, "roc_auc": True,
}

baselines = [m for m in method_order if m != "CellFlow"]

rows = []
for ct in ct_order:
    ct_data = df_mean[df_mean["cell_type_new"] == ct]
    row = {"cell_type": ct}
    for metric, title in all_metrics:
        cf_val = ct_data.loc[ct_data["model"] == "CellFlow", metric].values
        bl_vals = ct_data.loc[ct_data["model"].isin(baselines), metric].dropna().values
        if len(cf_val) == 0 or len(bl_vals) == 0:
            row[title] = np.nan
            continue
        cf_val = cf_val[0]
        if higher_better.get(metric, True):
            best_bl = bl_vals.max()
            row[title] = cf_val - best_bl
        else:
            best_bl = bl_vals.min()
            row[title] = best_bl - cf_val  # positive = CF is better
    rows.append(row)

df_heatmap = pd.DataFrame(rows).set_index("cell_type")

fig, ax = plt.subplots(figsize=(18, 9))
sns.heatmap(
    df_heatmap, annot=True, fmt=".3f", cmap="RdYlGn", center=0,
    linewidths=0.5, ax=ax, cbar_kws={"label": "CellFlow improvement over best baseline"},
    annot_kws={"size": 8},
)
ax.set_title("CellFlow improvement over best baseline (per cell type)", fontsize=14)
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=45, labelsize=9)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
fig.savefig(os.path.join(OUTDIR, "cell_eval_per_ct_heatmap.png"), dpi=150, bbox_inches="tight")
fig.savefig(os.path.join(OUTDIR, "cell_eval_per_ct_heatmap.pdf"), bbox_inches="tight")
plt.show()